# LongBook — Colab Chat
ブラウザーだけで起動する、個人用のLongBookチャットUIです。

**まずデモ/API接続で起動できます。** LongBookをColab内で実行する場合は、GPUランタイムを選択して下の `INSTALL_LOCAL` を有効にしてください。9B/27Bが無料Colabで必ず動くという保証はありません。独自カーネルの互換性、GPUの割当、メモリ容量に依存します。

認証付きGradio共有リンクを作ります。ログイン情報・APIキー・個人文書を含む出力を残したまま、このノートブックを公開しないでください。


In [ ]:
# @title 1. インストール
SOURCE_REF = "feat/portable-chat-harness" # @param {type:"string"}
INSTALL_LOCAL = False # @param {type:"boolean"}
INSTALL_4BIT = False # @param {type:"boolean"}
import re, subprocess, sys
if not re.fullmatch(r"[A-Za-z0-9_./-]+", SOURCE_REF):
    raise ValueError("SOURCE_REFにはブランチ・タグ・commit SHAを指定してください。")
extras = "[local,quantized]" if INSTALL_4BIT else ("[local]" if INSTALL_LOCAL else "")
requirement = f"longbook-harness{extras} @ git+https://github.com/vnlpscale/LongBook.git@{SOURCE_REF}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", requirement], check=True)
print("インストール完了。モデル重みはまだダウンロードしていません。")


In [ ]:
# @title 2. 認証付きチャットUIを起動
import os
os.environ["LONGBOOK_DATA_DIR"] = "/content/longbook-data"
os.environ["GRADIO_ANALYTICS_ENABLED"] = "False"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
if "demo" in globals():
    runtime.cancel()
    demo.close()
from longbook_harness.cli import launch_ui
# 個人用のランダムなパスワードが下に表示されます。ユーザー名は longbook です。
demo, runtime = launch_ui(share=True, inbrowser=False, prevent_thread_lock=True)


## 使い方
共有URLを開き、表示されたログイン情報でログインします。最初は接続確認デモです。

右上の「接続設定」でAPIまたはTransformersを選択し、「接続 / モデル読込」を押してください。外部Pythonコードを使うLongBookでは「revision確認」でSHAを取得し、モデルリポジトリのコードを確認してから実行許可を有効にします。4bitでは、インストール時に `INSTALL_4BIT=True`、UIの量子化で `4bit / CUDA` を選択します。

会話はブラウザー側、文書メモリは `/content/longbook-data` に保存します。**Colab終了時にランタイム上のメモリは失われます。** 必要な会話はUIからJSONに、メモリは次の任意セルでバックアップしてください。これはSQLiteの小規模メモリであり、1Tトークン向けの分散記憶ではありません。


In [ ]:
# @title 3. 任意：文書メモリをバックアップしてダウンロード
import sqlite3
from pathlib import Path
from google.colab import files
source = Path("/content/longbook-data/memory.sqlite3")
backup = Path("/content/longbook-memory-backup.sqlite3")
if not source.exists():
    raise FileNotFoundError("先にUIを起動してください。")
with sqlite3.connect(source) as src, sqlite3.connect(backup) as dst:
    src.backup(dst)
files.download(str(backup))


バックアップには登録した原文も含まれます。非公開で保管してください。
復元時はUIを停止し、バックアップを新しい空のデータディレクトリに `memory.sqlite3` という名前で置いてから起動します。
